# HARD Preprocessing Drill — "throw anything at me"

**Target: BCG Forward Deployed AI Scientist — CodeSignal (Sat).**
This is the harder follow-up to the Titanic drill. Same ARENA rules: read → fill the `# TODO` blanks → run → *only then* expand the Solution.

Where the first notebook was a clean ride, this one is built to **break your reflexes**. The dataset is synthetic and deliberately filthy:
* free-text numbers (`"  1,234.50 EUR "`), inconsistent casing/whitespace in categories
* two different date formats in one column + impossible dates
* a high-cardinality categorical (city) that one-hot can't handle
* heavy right-skew, negative sentinels (`-999`) standing in for missing
* **structured** missingness (income missing more often for one segment)
* two leakage traps you must spot and remove
* class imbalance (~12% positives)

If a cell errors on real-world junk, that's the lesson — assessments feed you exactly this. Build the muscle to clean it fast and leak-free.


## 0 — Setup & generate the messy dataset (just run this)

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

rng = np.random.default_rng(0)
N = 4000

seg = rng.choice(["retail", "sme", "corporate"], size=N, p=[0.6, 0.3, 0.1])
city_pool = [f"city_{i}" for i in range(120)]          # high cardinality
city = rng.choice(city_pool, size=N)

# skewed income; corporate earns more; structured missingness (sme hides income more)
income = np.exp(rng.normal(10.5, 0.6, N)) + (seg == "corporate") * 40000
miss_income = rng.random(N) < np.where(seg == "sme", 0.30, 0.08)
income[miss_income] = np.nan

age = rng.integers(18, 90, N).astype(float)
age[rng.random(N) < 0.03] = -999                        # sentinel for missing

# free-text money column with units, commas, whitespace, currency words
bal = rng.normal(5000, 3000, N).round(2)
balance_text = []
for b in bal:
    s = f"{abs(b):,.2f}"
    unit = rng.choice([" EUR", "€", " eur", ""], p=[0.4, 0.2, 0.2, 0.2])
    pad = rng.choice(["  ", " ", ""], p=[0.3, 0.4, 0.3])
    balance_text.append(f"{pad}{s}{unit}{pad}")

# messy categorical: case + whitespace noise
status_clean = rng.choice(["active", "churned", "dormant"], N, p=[0.7, 0.15, 0.15])
status = np.array([rng.choice([s, s.upper(), s.capitalize(), f" {s} "]) for s in status_clean], dtype=object)

# two date formats mixed + a few impossible
def make_date(i):
    y = rng.integers(2018, 2025); m = rng.integers(1, 13); d = rng.integers(1, 29)
    if rng.random() < 0.5:
        return f"{y}-{m:02d}-{d:02d}"          # ISO
    return f"{d:02d}/{m:02d}/{y}"              # DMY
signup = [make_date(i) for i in range(N)]
for j in rng.choice(N, 25, replace=False):
    signup[j] = rng.choice(["2024-13-01", "31/02/2023", "not_available", ""])

# leak #1: an id; leak #2: a column derived from the target
cust_id = np.arange(N)

# target: churn-ish, depends on income/seg/status
lin = (-(np.log(np.nan_to_num(income, nan=np.nanmedian(income))) - 10.5)
       + (status_clean == "churned") * 2.0 + (seg == "retail") * 0.4)
p = 1 / (1 + np.exp(-(lin - 1.7)))
y = (rng.random(N) < p).astype(int)
risk_note = np.where(y == 1, "FLAG", rng.choice(["", "ok", "review"], N))  # leak #2

df = pd.DataFrame({
    "cust_id": cust_id, "segment": seg, "city": city, "income": income,
    "age": age, "balance_text": balance_text, "status": status,
    "signup_date": signup, "risk_note": risk_note, "churned": y,
})
# scramble a few exact-duplicate rows in
df = pd.concat([df, df.iloc[:40]], ignore_index=True)
print(df.shape, "| positive rate:", round(df["churned"].mean(), 3))
df.head()

---
# Part 1 — See the mess clearly

You can't clean what you haven't characterized. The recon here is harder: dtypes lie (numbers stored as text), sentinels masquerade as values, and missingness is uneven across segments.

### Exercise - Recon that catches disguised problems

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Time: 5 min
> Skill: Data cleaning & preprocessing
> ```

Run the standard recon, then add two checks that catch *disguised* issues:
* exact-duplicate row count,
* for `age`, how many rows equal the sentinel `-999` (these are really missing, not real ages).


In [ ]:
print(df.shape)
df.info()
# TODO: duplicate rows
...
# TODO: count of the -999 sentinel in age
...

<details><summary>Solution</summary>

```python
print(df.shape)
df.info()
print("dup rows:", df.duplicated().sum())
print("age sentinels:", (df["age"] == -999).sum())
```

</details>

### Exercise - Quantify structured missingness

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Time: 6 min
> Skill: Data cleaning & preprocessing
> ```

Missingness in `income` is **not** random — it depends on `segment`. Show the **fraction of missing `income` per segment**. (If it varies by segment, that's MAR: an `income_missing` flag will carry signal, and you should impute *within* segment, not globally.)


In [ ]:
df.groupby(...)["income"].apply(lambda s: ...)

<details><summary>Solution</summary>

```python
df.groupby("segment")["income"].apply(lambda s: s.isna().mean()).round(3)
```

</details>

---
# Part 2 — Parsing filthy columns into clean types

This is the part people freeze on under time pressure. Three classic messes: money-as-text, sentinel-as-number, and mixed-format dates.

### Exercise - Parse free-text money into a float

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Time: 7 min
> Skill: Data cleaning & preprocessing
> ```

`balance_text` looks like `"  1,234.50 EUR "`, `"€987.00"`, `"5,000.00 eur"`. Turn it into a clean float column `balance`.

Strategy: strip whitespace → remove currency tokens (`EUR`, `eur`, `€`) → remove thousands commas → `astype(float)`. Do it with vectorized `str` ops + one regex, no Python loop.


In [ ]:
df["balance"] = (
    df["balance_text"]
      .str.strip()
      .str.replace(r"...", "", regex=True)   # currency words/symbols
      .str.replace(",", "", regex=False)     # thousands sep
      .str.strip()
      .astype(float)
)
df[["balance_text", "balance"]].head()

<details><summary>Solution</summary>

```python
df["balance"] = (
    df["balance_text"]
      .str.strip()
      .str.replace(r"(?i)\s*(eur|€)\s*", "", regex=True)
      .str.replace(",", "", regex=False)
      .str.strip()
      .astype(float)
)
df[["balance_text", "balance"]].head()
```

</details>

### Exercise - Sentinel -> NaN, then sanity-bound

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Time: 4 min
> Skill: Data cleaning & preprocessing
> ```

In `age`, the value `-999` means missing. Replace it with `np.nan`. Then guard against impossible ages by setting anything outside `[0, 120]` to NaN as well.


In [ ]:
df["age"] = df["age"].replace(..., np.nan)
df.loc[(df["age"] < ...) | (df["age"] > ...), "age"] = np.nan
df["age"].describe()

<details><summary>Solution</summary>

```python
df["age"] = df["age"].replace(-999, np.nan)
df.loc[(df["age"] < 0) | (df["age"] > 120), "age"] = np.nan
df["age"].describe()
```

</details>

### Exercise - Normalize a messy categorical

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Time: 4 min
> Skill: Data cleaning & preprocessing
> ```

`status` contains `"active"`, `"ACTIVE"`, `"Active"`, `" active "`. Collapse them to a clean lowercase, stripped category. Confirm you end with exactly 3 unique values.


In [ ]:
df["status"] = df["status"].str....
print(df["status"].unique())

<details><summary>Solution</summary>

```python
df["status"] = df["status"].str.strip().str.lower()
print(df["status"].unique())
```

</details>

### Exercise - Parse mixed-format dates safely

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Time: 8 min
> Skill: Data cleaning & preprocessing
> ```

`signup_date` mixes ISO (`2024-03-09`) and DMY (`09/03/2024`), plus junk (`"2024-13-01"`, `"not_available"`, `""`). Produce a real datetime column `signup_dt` where unparseable entries become `NaT` (never crash).

Approach: parse the two formats separately with `errors="coerce"` and combine (take the first that parsed). Then derive `tenure_days` from a fixed reference date `2025-01-01` (don't use "today" — assessments need determinism).


In [ ]:
iso = pd.to_datetime(df["signup_date"], format=..., errors="coerce")
dmy = pd.to_datetime(df["signup_date"], format=..., errors="coerce")
df["signup_dt"] = iso.fillna(...)
ref = pd.Timestamp("2025-01-01")
df["tenure_days"] = (ref - df["signup_dt"]).dt.days
print("unparsed:", df["signup_dt"].isna().sum())
df[["signup_date", "signup_dt", "tenure_days"]].head()

<details><summary>Solution</summary>

```python
iso = pd.to_datetime(df["signup_date"], format="%Y-%m-%d", errors="coerce")
dmy = pd.to_datetime(df["signup_date"], format="%d/%m/%Y", errors="coerce")
df["signup_dt"] = iso.fillna(dmy)
ref = pd.Timestamp("2025-01-01")
df["tenure_days"] = (ref - df["signup_dt"]).dt.days
print("unparsed:", df["signup_dt"].isna().sum())
df[["signup_date", "signup_dt", "tenure_days"]].head()
```

</details>

---
# Part 3 — Distribution fixes: skew, scaling choices

Real numeric features are rarely Gaussian. Knowing *which* transform to reach for is the skill.

### Exercise - Diagnose & fix skew with a log1p

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Time: 5 min
> Skill: Data cleaning & preprocessing
> ```

`income` is heavily right-skewed. Print its skew, apply `np.log1p` to a new column `income_log`, and print the new skew. (Rule: |skew| > 1 → consider a log/Box-Cox; log1p is safe for zeros.)


In [ ]:
print("skew before:", round(df["income"].skew(), 2))
df["income_log"] = ...
print("skew after :", round(df["income_log"].skew(), 2))

<details><summary>Solution</summary>

```python
print("skew before:", round(df["income"].skew(), 2))
df["income_log"] = np.log1p(df["income"])
print("skew after :", round(df["income_log"].skew(), 2))
```

</details>

### Exercise - Box-Cox vs Yeo-Johnson (when log isn't enough)

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Time: 6 min
> Skill: Data cleaning & preprocessing
> ```

`balance` can be ~0 or negative-ish after noise, so Box-Cox (needs strictly positive) is risky — **Yeo-Johnson** handles any sign. Use sklearn's `PowerTransformer(method="yeo-johnson")` on `balance` (fit on non-null values) and report the skew before/after. Note: in a real run you'd fit this inside the pipeline on train only.


In [ ]:
from sklearn.preprocessing import PowerTransformer
vals = df["balance"].dropna().values.reshape(-1, 1)
pt = PowerTransformer(method=...)
trans = pt.fit_transform(vals).ravel()
print("skew before:", round(stats.skew(vals.ravel()), 2),
      "after:", round(stats.skew(trans), 2))

<details><summary>Solution</summary>

```python
from sklearn.preprocessing import PowerTransformer
vals = df["balance"].dropna().values.reshape(-1, 1)
pt = PowerTransformer(method="yeo-johnson")
trans = pt.fit_transform(vals).ravel()
print("skew before:", round(stats.skew(vals.ravel()), 2),
      "after:", round(stats.skew(trans), 2))
```

</details>

---
# Part 4 — High-cardinality categoricals (where one-hot dies)

`city` has 120 levels. One-hot → 120 sparse columns, slow and overfit-prone. Two pro moves: **frequency encoding** and **leak-free target encoding**.

### Exercise - Frequency / count encoding

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Time: 4 min
> Skill: Data cleaning & preprocessing
> ```

Replace `city` with how often each city appears (a cheap, often-effective encoding). Create `city_freq` = the normalized value count mapped back to each row.


In [ ]:
freq = df["city"].value_counts(normalize=True)
df["city_freq"] = df["city"].map(...)
df[["city", "city_freq"]].head()

<details><summary>Solution</summary>

```python
freq = df["city"].value_counts(normalize=True)
df["city_freq"] = df["city"].map(freq)
df[["city", "city_freq"]].head()
```

</details>

### Exercise - Leak-free target encoding with K-fold + smoothing

> ```yaml
> Difficulty: 🔴🔴🔴🔴🔴
> Time: 12 min
> Skill: ML & predictive modeling
> ```

The classic trap: encoding a category by its mean target on the **whole** dataset leaks the target. Do it the right way — **out-of-fold** target encoding with **smoothing** toward the global mean.

For each fold, compute category means on the *other* folds and map onto the held-out fold. Smoothing: `enc = (count*cat_mean + m*global_mean) / (count + m)` with `m=20`.

Fill `city_te`. This is a senior-level signal — getting the out-of-fold logic right is the whole point.


In [ ]:
from sklearn.model_selection import KFold

def kfold_target_encode(frame, col, target, n_splits=5, m=20, seed=0):
    global_mean = frame[target].mean()
    out = pd.Series(index=frame.index, dtype=float)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr_idx, val_idx in kf.split(frame):
        tr = frame.iloc[tr_idx]
        stats_ = tr.groupby(col)[target].agg(["mean", "count"])
        smooth = (stats_["count"] * stats_["mean"] + m * global_mean) / (stats_["count"] + m)
        out.iloc[val_idx] = frame.iloc[val_idx][col].map(...).fillna(...)
    return out

df["city_te"] = kfold_target_encode(df, "city", "churned")
df[["city", "city_te"]].head()

<details><summary>Solution</summary>

```python
from sklearn.model_selection import KFold

def kfold_target_encode(frame, col, target, n_splits=5, m=20, seed=0):
    global_mean = frame[target].mean()
    out = pd.Series(index=frame.index, dtype=float)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr_idx, val_idx in kf.split(frame):
        tr = frame.iloc[tr_idx]
        stats_ = tr.groupby(col)[target].agg(["mean", "count"])
        smooth = (stats_["count"] * stats_["mean"] + m * global_mean) / (stats_["count"] + m)
        out.iloc[val_idx] = frame.iloc[val_idx][col].map(smooth).fillna(global_mean)
    return out

df["city_te"] = kfold_target_encode(df, "city", "churned")
df[["city", "city_te"]].head()
```

</details>

---
# Part 5 — Spot the leaks (this separates seniors from juniors)

Before modeling you must remove anything that (a) is an identifier with no signal, or (b) encodes the target. Missing this tanks you in the interview even if your accuracy looks great.

### Exercise - Find and justify the leaks

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Time: 6 min
> Skill: ML & predictive modeling
> ```

Two columns must go:
* `cust_id` — a unique identifier; check it's ~unique and carries no signal.
* `risk_note` — inspect its relationship to `churned` via a crosstab. If one value appears almost only for churners, it's a leak (it was created from the label).

Print the evidence, then drop both. Also drop `balance_text`/`signup_date` (raw text we've parsed) and `income` if you keep `income_log`.


In [ ]:
print("cust_id unique frac:", df["cust_id"].nunique() / len(df))
display(pd.crosstab(df["risk_note"], df["churned"], normalize="index").round(2))
leak_cols = [...]
df = df.drop(columns=leak_cols)
df.columns.tolist()

<details><summary>Solution</summary>

```python
print("cust_id unique frac:", df["cust_id"].nunique() / len(df))
display(pd.crosstab(df["risk_note"], df["churned"], normalize="index").round(2))
leak_cols = ["cust_id", "risk_note", "balance_text", "signup_date", "signup_dt"]
df = df.drop(columns=leak_cols)
df.columns.tolist()
```

</details>

---
# Part 6 — A custom transformer + full leak-proof pipeline

The grown-up version of the Titanic pipeline: a **custom `FunctionTransformer`/`BaseEstimator`** step, an `income_missing` flag added inside the pipeline, ColumnTransformer over mixed types, and class-imbalance handling via `class_weight`.

### Exercise - Custom sklearn transformer (add a missing-indicator)

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Time: 9 min
> Skill: ML & predictive modeling
> ```

Write a `BaseEstimator/TransformerMixin` class `MissingIndicator2` that, given numeric columns, **appends a 0/1 missing-flag column for each**. It must implement `fit` (no-op, return self) and `transform` (return original + flags). This is the pattern for any bespoke cleaning step that has to live inside a pipeline.


In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class MissingIndicator2(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = pd.DataFrame(X).copy()
        flags = ...   # 0/1 isna for each column, suffixed _missing
        return pd.concat([X, flags], axis=1)

mi = MissingIndicator2()
out = mi.fit_transform(df[["income", "age"]])
out.head()

<details><summary>Solution</summary>

```python
from sklearn.base import BaseEstimator, TransformerMixin

class MissingIndicator2(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = pd.DataFrame(X).copy()
        flags = X.isna().astype(int).add_suffix("_missing")
        return pd.concat([X, flags], axis=1)

mi = MissingIndicator2()
out = mi.fit_transform(df[["income", "age"]])
out.head()
```

</details>

### Exercise - Assemble the full pipeline & evaluate on imbalanced data

> ```yaml
> Difficulty: 🔴🔴🔴🔴🔴
> Time: 12 min
> Skill: ML & predictive modeling
> ```

Build the end-to-end pipeline on the cleaned `df`:
* numeric (`income_log`, `age`, `balance`, `tenure_days`, `city_freq`, `city_te`): median impute → standard scale
* low-card categorical (`segment`, `status`): most-frequent impute → one-hot
* model: `LogisticRegression(class_weight="balanced", max_iter=1000)` — the `class_weight` is how you handle the 12% imbalance without resampling.

Split with `stratify`, fit, and report **ROC-AUC and the classification report** (accuracy is meaningless at 12% positives — say so).


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report

y = df["churned"]
X = df.drop(columns=["churned", "city"])   # drop raw city; keep its encodings

num_cols = ["income_log", "age", "balance", "tenure_days", "city_freq", "city_te"]
cat_cols = ["segment", "status"]

num = Pipeline([...])
cat = Pipeline([...])
pre = ColumnTransformer([...])
clf = Pipeline([("pre", pre), ("lr", LogisticRegression(class_weight=..., max_iter=1000))])

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=..., random_state=0)
clf.fit(X_tr, y_tr)
proba = clf.predict_proba(X_te)[:, 1]
print("ROC-AUC:", round(roc_auc_score(y_te, proba), 3))
print(classification_report(y_te, clf.predict(X_te)))

<details><summary>Solution</summary>

```python
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report

y = df["churned"]
X = df.drop(columns=["churned", "city"])

num_cols = ["income_log", "age", "balance", "tenure_days", "city_freq", "city_te"]
cat_cols = ["segment", "status"]

num = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("sc", StandardScaler()),
])
cat = Pipeline([
    ("imp", SimpleImputer(strategy="most_frequent")),
    ("oh", OneHotEncoder(handle_unknown="ignore")),
])
pre = ColumnTransformer([
    ("num", num, num_cols),
    ("cat", cat, cat_cols),
])
clf = Pipeline([("pre", pre), ("lr", LogisticRegression(class_weight="balanced", max_iter=1000))])

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
clf.fit(X_tr, y_tr)
proba = clf.predict_proba(X_te)[:, 1]
print("ROC-AUC:", round(roc_auc_score(y_te, proba), 3))
print(classification_report(y_te, clf.predict(X_te)))
```

</details>

---
# Part 7 — Threshold tuning (the step everyone forgets)

A classifier outputs probabilities; the 0.5 default is rarely optimal on imbalanced data. Pick a threshold from the precision-recall trade-off.

### Exercise - Choose a threshold that maximizes F1

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Time: 7 min
> Skill: ML & predictive modeling
> ```

Using `precision_recall_curve` on the test probabilities, compute F1 at every threshold and report the threshold that maximizes it, plus that F1. Compare to F1 at the default 0.5.


In [ ]:
from sklearn.metrics import precision_recall_curve, f1_score
prec, rec, thr = precision_recall_curve(y_te, proba)
f1s = 2 * prec * rec / (prec + rec + 1e-9)
best = ...  # index of max f1 (note: thr has len-1 vs prec/rec)
print("best thr:", round(thr[best], 3), "F1:", round(f1s[best], 3))
print("F1 @0.5 :", round(f1_score(y_te, (proba >= 0.5).astype(int)), 3))

<details><summary>Solution</summary>

```python
from sklearn.metrics import precision_recall_curve, f1_score
prec, rec, thr = precision_recall_curve(y_te, proba)
f1s = 2 * prec * rec / (prec + rec + 1e-9)
best = np.argmax(f1s[:-1])
print("best thr:", round(thr[best], 3), "F1:", round(f1s[best], 3))
print("F1 @0.5 :", round(f1_score(y_te, (proba >= 0.5).astype(int)), 3))
```

</details>

---
# Part 8 — Hard cheat sheet

**Disguised-problem checklist (run mentally on every column):**
* number stored as text? → `str` clean + `astype`
* sentinel (`-999`, `0`, `9999`) hiding missingness? → `replace(..., nan)`
* category with case/whitespace variants? → `.str.strip().str.lower()`
* dates in >1 format? → parse each `format=` with `errors="coerce"`, `fillna` across
* high cardinality (>~15 levels)? → frequency or **out-of-fold** target encoding, never whole-data target mean
* skew |·|>1? → `log1p` (≥0) or `PowerTransformer` yeo-johnson (any sign)
* structured missingness? → add `_missing` flag + impute within group

**Leakage kill-list:** ids, post-outcome fields, anything computed from the label, target-encoded-on-full-data, scaler/imputer fit on full data.

**Imbalance:** don't trust accuracy; use ROC-AUC / PR-AUC / F1; `class_weight="balanced"` or resample; tune the threshold.

**Out-of-fold target encoding** (type it blind — the highest-value snippet here):
```python
for tr, val in KFold(5, shuffle=True, random_state=0).split(df):
    s = df.iloc[tr].groupby(col)[y].agg(["mean","count"])
    sm = (s["count"]*s["mean"] + m*gmean)/(s["count"]+m)
    out.iloc[val] = df.iloc[val][col].map(sm).fillna(gmean)
```

Tomorrow this should feel like the *easy* notebook. That's the goal.
